In [ ]:
import torch
from braindecode.models import EEGNet
from braindecode import EEGClassifier
import torch.nn as nn
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV

In [ ]:
# config
DATA_FOLDER = "data/"
DATA_SESSION = "3-4/joshfoot/"
SESSIONS = [7, 8, 9]
CHANNELS = [2, 3, 4, 6] # in theory these are the only ones that should matter

In [ ]:
# load filtered data
alltrials = np.empty((0, 2))
for session in SESSIONS:
    filtered_session = np.load(f"{DATA_FOLDER}{DATA_SESSION}filtered-session-{session}.npy", allow_pickle=True)
    for i in range(len(filtered_session)):
        filtered_session[i][1] = filtered_session[i][1][CHANNELS, 62:-62]
    alltrials = np.concatenate((alltrials, filtered_session), axis=0)

In [ ]:
# set up csp and lda labels
labels = np.array([1 if trial[0] == 'clench right foot' else 0 for trial in alltrials])
eeg = np.array([trial[1] for trial in alltrials])

print(labels.shape)
print(eeg.shape)

In [ ]:
n_ch, n_t = eeg.shape[1], eeg.shape[2]
eeg_f32 = eeg.astype(np.float32)

In [ ]:
# baseline EEGNet
clf = EEGClassifier(
    module=EEGNet,
    module__n_chans=n_ch,
    module__n_outputs=2,
    module__n_times=n_t,
    module__kernel_length=125,
    module__drop_prob=0.5,
    optimizer=torch.optim.Adam,
    optimizer__lr=1e-3,
    optimizer__weight_decay=1e-4,
    criterion=nn.CrossEntropyLoss,
    max_epochs=60,
    batch_size=8,
    train_split=None,
    callbacks=[],
    verbose=0,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, eeg_f32, labels, cv=cv, scoring='accuracy')
print(f"fold scores: {scores}")

In [ ]:
# grid search
param_grid = {
    'module__kernel_length': [32, 64, 125, 250],
    'module__F1': [4, 8, 16],
    'module__D': [1, 2],
    'module__drop_prob': [0.25, 0.5, 0.75],
    'optimizer__lr': [5e-4, 1e-3],
    'batch_size': [4, 8, 16],
}

gs = GridSearchCV(
    clf, param_grid,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy',
    refit=True,
    verbose=2,
    error_score='raise',
)

gs.fit(eeg_f32, labels)
print(f"best score: {gs.best_score_}")
print(f"best params: {gs.best_params_}")